In [1]:
print ("Debut du TP3 Retail Intelligence")

Debut du TP3 Retail Intelligence


In [2]:
import sys
print(sys.executable)

/home/ubuntu/retail-intelligence/tp3/.venv/bin/python


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TP3-Retail-Intelligence")
    .master("local[*]")
    .getOrCreate()
)

print("Version Spark :", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 14:31:08 WARN Utils: Your hostname, retail, resolves to a loopback address: 127.0.1.1; using 192.168.252.2 instead (on interface eth0)
26/11/18 14:31:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 14:31:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Version Spark : 4.3.0


In [7]:
df = (
    spark.read
    .option("header", False)
    .option("inferSchema", True)
    .option("sep", "|")
    .csv("hdfs://localhost:9000/retail-lake/raw/tpcxbb/initial/store_sales.dat")
)

# Afficher les premières lignes
df.show(5)

+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|  _c0|  _c1|  _c2|  _c3|    _c4| _c5|  _c6|_c7|_c8|_c9|_c10| _c11|  _c12| _c13|   _c14|   _c15|   _c16|   _c17|  _c18|  _c19|   _c20|   _c21|   _c22|
+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|36890|71480|10001|44404| 493944|2898|44093| 10|294|  0|  31| 18.0| 30.78|23.08|  238.7| 715.48|  558.0| 954.18| 21.46|   0.0| 715.48| 736.94| 157.48|
|36890|71480|10000|44404| 493944|2898|44093| 10|279|  0|  54|99.21|148.82|83.34|3535.92|4500.36|5357.34|8036.28|225.02|   0.0|4500.36|4725.38|-856.98|
|36890|71480|15556|44404| 493944|2898|44093| 10|114|  0|  39|79.49| 94.59|83.24| 442.65|3246.36|3100.11|3689.01|184.07|616.81|2629.55|2813.62|-470.56|
|36890|45392| 4597|89832|1774847|6266|16285|  4|285|  1|  20|74.38|136.12|133.4|   54.4| 2668.

In [8]:
# Afficher les colonnes et leurs types
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: integer (nullable = true)
 |-- _c2: integer (nullable = true)
 |-- _c3: integer (nullable = true)
 |-- _c4: integer (nullable = true)
 |-- _c5: integer (nullable = true)
 |-- _c6: integer (nullable = true)
 |-- _c7: integer (nullable = true)
 |-- _c8: integer (nullable = true)
 |-- _c9: integer (nullable = true)
 |-- _c10: integer (nullable = true)
 |-- _c11: double (nullable = true)
 |-- _c12: double (nullable = true)
 |-- _c13: double (nullable = true)
 |-- _c14: double (nullable = true)
 |-- _c15: double (nullable = true)
 |-- _c16: double (nullable = true)
 |-- _c17: double (nullable = true)
 |-- _c18: double (nullable = true)
 |-- _c19: double (nullable = true)
 |-- _c20: double (nullable = true)
 |-- _c21: double (nullable = true)
 |-- _c22: double (nullable = true)



In [9]:
df_test = df.limit(5)

df_test.write.mode("overwrite").parquet(
    "hdfs://localhost:9000/retail-lake/tp3/test"
)

df_parquet = spark.read.parquet(
    "hdfs://localhost:9000/retail-lake/tp3/test"
)

df_parquet.show()

+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|  _c0|  _c1|  _c2|  _c3|    _c4| _c5|  _c6|_c7|_c8|_c9|_c10| _c11|  _c12| _c13|   _c14|   _c15|   _c16|   _c17|  _c18|  _c19|   _c20|   _c21|   _c22|
+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+-----+-------+-------+-------+-------+------+------+-------+-------+-------+
|36890|71480|10001|44404| 493944|2898|44093| 10|294|  0|  31| 18.0| 30.78|23.08|  238.7| 715.48|  558.0| 954.18| 21.46|   0.0| 715.48| 736.94| 157.48|
|36890|71480|10000|44404| 493944|2898|44093| 10|279|  0|  54|99.21|148.82|83.34|3535.92|4500.36|5357.34|8036.28|225.02|   0.0|4500.36|4725.38|-856.98|
|36890|71480|15556|44404| 493944|2898|44093| 10|114|  0|  39|79.49| 94.59|83.24| 442.65|3246.36|3100.11|3689.01|184.07|616.81|2629.55|2813.62|-470.56|
|36890|45392| 4597|89832|1774847|6266|16285|  4|285|  1|  20|74.38|136.12|133.4|   54.4| 2668.

In [10]:
print("Mode :", spark.sparkContext.master)
print("Coeurs utilisés :", spark.sparkContext.defaultParallelism)

Mode : local[*]
Coeurs utilisés : 4


In [11]:
chemin_source = "hdfs://localhost:9000/retail-lake/raw/tpcxbb/initial/date_dim.dat"

df_dates = (
    spark.read
    .option("header", False)
    .option("inferSchema", True)
    .option("sep", "|")
    .csv(chemin_source)
)

# Observer les cplonnes et leurs types
df_dates.printSchema()

# Afficher les 10 premières lignes
df_dates.show(10, truncate=False)

# Compter les lignes
print("Nombre de lignes :", df_dates.count())

root
 |-- _c0: integer (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: date (nullable = true)
 |-- _c3: integer (nullable = true)
 |-- _c4: integer (nullable = true)
 |-- _c5: integer (nullable = true)
 |-- _c6: integer (nullable = true)
 |-- _c7: integer (nullable = true)
 |-- _c8: integer (nullable = true)
 |-- _c9: integer (nullable = true)
 |-- _c10: integer (nullable = true)
 |-- _c11: integer (nullable = true)
 |-- _c12: integer (nullable = true)
 |-- _c13: integer (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _c16: string (nullable = true)
 |-- _c17: string (nullable = true)
 |-- _c18: string (nullable = true)
 |-- _c19: integer (nullable = true)
 |-- _c20: integer (nullable = true)
 |-- _c21: integer (nullable = true)
 |-- _c22: integer (nullable = true)
 |-- _c23: string (nullable = true)
 |-- _c24: string (nullable = true)
 |-- _c25: string (nullable = true)
 |-- _c26: string (nullable = true)
 |-- _c27: string (

26/11/18 15:45:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Nombre de lignes : 109573


In [12]:
print("Nombre de lignes :", df.count())

Nombre de lignes : 667579


In [13]:
# Noms des colonnes : specification TPCx-BB 1.7.0, dans le même ordre qu'au Jour 1
colonnes_ventes = [
    "ss_sold_date_sk", "ss_sold_time_sk", "ss_item_sk", "ss_customer_sk",
    "ss_cdemo_sk", "ss_hdemo_sk", "ss_addr_sk", "ss_store_sk", "ss_promo_sk",
    "ss_ticket_number", "ss_quantity", "ss_wholesale_cost", "ss_list_price",
    "ss_sales_price", "ss_ext_discount_amt", "ss_ext_sales_price",
    "ss_ext_wholesale_cost", "ss_ext_list_price", "ss_ext_tax",
    "ss_coupon_amt", "ss_net_paid", "ss_net_paid_inc_tax", "ss_net_profit"
]

colonnes_dates = [
    "d_date_sk", "d_date_id", "d_date", "d_month_seq", "d_week_seq",
    "d_quarter_seq", "d_year", "d_dow", "d_moy", "d_dom", "d_qoy",
    "d_fy_year", "d_fy_quarter_seq", "d_fy_week_seq", "d_day_name",
    "d_quarter_name", "d_holiday", "d_weekend", "d_following_holiday",
    "d_first_dom", "d_last_dom", "d_same_day_ly", "d_same_day_lq",
    "d_current_day", "d_current_week", "d_current_month",
    "d_current_quarter", "d_current_year"
]

df_ventes = df.toDF(*colonnes_ventes)
df_dates = df_dates.toDF(*colonnes_dates)

df_ventes.printSchema()
df_dates.printSchema()

root
 |-- ss_sold_date_sk: integer (nullable = true)
 |-- ss_sold_time_sk: integer (nullable = true)
 |-- ss_item_sk: integer (nullable = true)
 |-- ss_customer_sk: integer (nullable = true)
 |-- ss_cdemo_sk: integer (nullable = true)
 |-- ss_hdemo_sk: integer (nullable = true)
 |-- ss_addr_sk: integer (nullable = true)
 |-- ss_store_sk: integer (nullable = true)
 |-- ss_promo_sk: integer (nullable = true)
 |-- ss_ticket_number: integer (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: double (nullable = true)
 |-- ss_list_price: double (nullable = true)
 |-- ss_sales_price: double (nullable = true)
 |-- ss_ext_discount_amt: double (nullable = true)
 |-- ss_ext_sales_price: double (nullable = true)
 |-- ss_ext_wholesale_cost: double (nullable = true)
 |-- ss_ext_list_price: double (nullable = true)
 |-- ss_ext_tax: double (nullable = true)
 |-- ss_coupon_amt: double (nullable = true)
 |-- ss_net_paid: double (nullable = true)
 |-- ss_net_paid_inc_tax

In [14]:
from pyspark.sql import functions as F

for nom_colonne in df_ventes.columns:
    nb_vides = df_ventes.filter(
        F.col(nom_colonne).isNull()
    ).count()
    print(nom_colonne, ":", nb_vides, "valeur(s) nulle(s)")

ss_sold_date_sk : 0 valeur(s) nulle(s)
ss_sold_time_sk : 0 valeur(s) nulle(s)
ss_item_sk : 0 valeur(s) nulle(s)
ss_customer_sk : 0 valeur(s) nulle(s)
ss_cdemo_sk : 238 valeur(s) nulle(s)
ss_hdemo_sk : 295 valeur(s) nulle(s)
ss_addr_sk : 448 valeur(s) nulle(s)
ss_store_sk : 0 valeur(s) nulle(s)
ss_promo_sk : 158 valeur(s) nulle(s)
ss_ticket_number : 0 valeur(s) nulle(s)
ss_quantity : 0 valeur(s) nulle(s)
ss_wholesale_cost : 0 valeur(s) nulle(s)
ss_list_price : 0 valeur(s) nulle(s)
ss_sales_price : 0 valeur(s) nulle(s)
ss_ext_discount_amt : 0 valeur(s) nulle(s)
ss_ext_sales_price : 0 valeur(s) nulle(s)
ss_ext_wholesale_cost : 0 valeur(s) nulle(s)
ss_ext_list_price : 0 valeur(s) nulle(s)
ss_ext_tax : 0 valeur(s) nulle(s)
ss_coupon_amt : 0 valeur(s) nulle(s)
ss_net_paid : 0 valeur(s) nulle(s)
ss_net_paid_inc_tax : 0 valeur(s) nulle(s)
ss_net_profit : 0 valeur(s) nulle(s)


In [15]:
for nom_colonne in df_dates.columns:
    nb_vides = df_dates.filter(
        F.col(nom_colonne).isNull()
    ).count()
    print(nom_colonne, ":", nb_vides, "valeur(s) nulle(s)")

d_date_sk : 0 valeur(s) nulle(s)
d_date_id : 0 valeur(s) nulle(s)
d_date : 0 valeur(s) nulle(s)
d_month_seq : 0 valeur(s) nulle(s)
d_week_seq : 0 valeur(s) nulle(s)
d_quarter_seq : 0 valeur(s) nulle(s)
d_year : 0 valeur(s) nulle(s)
d_dow : 0 valeur(s) nulle(s)
d_moy : 0 valeur(s) nulle(s)
d_dom : 0 valeur(s) nulle(s)
d_qoy : 0 valeur(s) nulle(s)
d_fy_year : 0 valeur(s) nulle(s)
d_fy_quarter_seq : 0 valeur(s) nulle(s)
d_fy_week_seq : 0 valeur(s) nulle(s)
d_day_name : 0 valeur(s) nulle(s)
d_quarter_name : 0 valeur(s) nulle(s)
d_holiday : 0 valeur(s) nulle(s)
d_weekend : 0 valeur(s) nulle(s)
d_following_holiday : 0 valeur(s) nulle(s)
d_first_dom : 0 valeur(s) nulle(s)
d_last_dom : 0 valeur(s) nulle(s)
d_same_day_ly : 0 valeur(s) nulle(s)
d_same_day_lq : 0 valeur(s) nulle(s)
d_current_day : 0 valeur(s) nulle(s)
d_current_week : 0 valeur(s) nulle(s)
d_current_month : 0 valeur(s) nulle(s)
d_current_quarter : 0 valeur(s) nulle(s)
d_current_year : 0 valeur(s) nulle(s)


In [19]:
print("Nombre de lignes 'doublon' dans store_sales :", df_ventes.count()-df_ventes.dropDuplicates().count())
print("Nombre de lignes 'doublon' dans date_dim :", df_dates.count()-df_dates.dropDuplicates().count())

Nombre de lignes 'doublon' dans store_sales : 0
Nombre de lignes 'doublon' dans date_dim : 0


In [21]:
# Clé candidate de date_dim
print("d_date_sk en double :", df_dates.count() - df_dates.select("d_date_sk").distinct().count())
print("d_date en double :", df_dates.count() - df_dates.select("d_date").distinct().count())

# Clé candidate de store_sales : couple ticket + produit
couples_en_double = (
    df_ventes.groupBy("ss_ticket_number", "ss_item_sk")
    .count()
    .filter(F.col("count") > 1)
)
print("couples ticket + produit en double :", couples_en_double.count())

d_date_sk en double : 0
d_date en double : 0
couples ticket + produit en double : 4038


In [23]:
df_ventes.select(
    "ss_quantity", "ss_sales_price", "ss_ext_sales_price",
    "ss_coupon_amt", "ss_net_paid", "ss_net_profit"
).describe().show()

+-------+------------------+-----------------+------------------+-----------------+------------------+------------------+
|summary|       ss_quantity|   ss_sales_price|ss_ext_sales_price|    ss_coupon_amt|       ss_net_paid|     ss_net_profit|
+-------+------------------+-----------------+------------------+-----------------+------------------+------------------+
|  count|            667579|           667579|            667579|           667579|            667579|            667579|
|   mean|50.564014146640325| 37.8636970605728|1913.2177683989435|189.7819108150499|1723.4358575838912|-827.6973583950435|
| stddev|  28.8808956685929|34.39915676234276|2281.7549042316823|725.3157530839015|2167.8986164781318|1727.4881593984812|
|    min|                 1|              0.0|               0.0|              0.0|               0.0|           -9866.6|
|    max|               100|            199.3|           18745.0|         14371.66|           18745.0|            9134.0|
+-------+---------------

In [24]:
for nom_colonne in ["ss_quantity", "ss_sales_price", "ss_ext_sales_price",
                    "ss_coupon_amt", "ss_net_paid", "ss_net_profit"]:
    nb_negatifs = df_ventes.filter(F.col(nom_colonne) < 0).count()
    print(nom_colonne, ":", nb_negatifs, "valeur(s) négative(s)")

ss_quantity : 0 valeur(s) négative(s)
ss_sales_price : 0 valeur(s) négative(s)
ss_ext_sales_price : 0 valeur(s) négative(s)
ss_coupon_amt : 0 valeur(s) négative(s)
ss_net_paid : 0 valeur(s) négative(s)
ss_net_profit : 495971 valeur(s) négative(s)


In [25]:
bornes = df_ventes.agg(
    F.min("ss_sold_date_sk").alias("min_sk"),
    F.max("ss_sold_date_sk").alias("max_sk")
).first()

df_dates.filter(
    F.col("d_date_sk").isin(bornes["min_sk"], bornes["max_sk"])
).select("d_date_sk", "d_date").show()

df_dates.agg(F.min("d_date"), F.max("d_date")).show()

+---------+----------+
|d_date_sk|    d_date|
+---------+----------+
|    36890|2001-01-01|
|    38697|2005-12-13|
+---------+----------+

+-----------+-----------+
|min(d_date)|max(d_date)|
+-----------+-----------+
| 1900-01-01| 2199-12-31|
+-----------+-----------+



In [26]:
print("lignes à prix de vente nul :", df_ventes.filter(F.col("ss_sales_price") == 0).count())

lignes à prix de vente nul : 3324
